## Gmail API + OpenAI — batch classification (`gmail_batch.ipynb`)

Same pipeline as `gmail.ipynb`, with two batching layers:

- **Gmail**: full messages are fetched with **HTTP batch** (`GMAIL_FETCH_BATCH_SIZE`, default 50 per round trip) instead of one `get` per email.
- **OpenAI**: several emails per chat request (`OPENAI_CLASSIFY_BATCH_SIZE` / `CLASSIFY_BATCH_SIZE`).

If the model skips an id in a batch response, that message falls back to a **single-email** classification call.


In [1]:
# If you're in a fresh environment, install dependencies:
# (You can also: pip install -r requirements.txt)

%pip -q install -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

CLIENT_SECRETS_FILE = os.getenv("GOOGLE_CLIENT_SECRETS_FILE", "credentials.json")
TOKEN_FILE = os.getenv("GMAIL_TOKEN_FILE", "token.json")
SCOPES = os.getenv("GMAIL_SCOPES", "https://www.googleapis.com/auth/gmail.readonly").split()

print("CLIENT_SECRETS_FILE:", CLIENT_SECRETS_FILE)
print("TOKEN_FILE:", TOKEN_FILE)
print("SCOPES:", SCOPES)

if not Path(CLIENT_SECRETS_FILE).exists():
    raise FileNotFoundError(
        f"Missing {CLIENT_SECRETS_FILE}. Download OAuth client secrets JSON from Google Cloud Console "
        f"and save it next to this notebook (or update GOOGLE_CLIENT_SECRETS_FILE in .env)."
    )


CLIENT_SECRETS_FILE: client_secret.json
TOKEN_FILE: token.json
SCOPES: ['https://www.googleapis.com/auth/gmail.readonly']


In [3]:
def get_gmail_service(*, client_secrets_file: str, token_file: str, scopes: list[str]):
    creds = None
    token_path = Path(token_file)

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(token_file, scopes)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secrets_file, scopes)
            creds = flow.run_local_server(port=0)

        token_path.write_text(creds.to_json(), encoding="utf-8")

    return build("gmail", "v1", credentials=creds)


service = get_gmail_service(
    client_secrets_file=CLIENT_SECRETS_FILE,
    token_file=TOKEN_FILE,
    scopes=SCOPES,
)

print("Gmail service created.")


Gmail service created.


In [23]:
def _header(headers: list[dict], name: str) -> str | None:
    name_lower = name.lower()
    for h in headers or []:
        if (h.get("name") or "").lower() == name_lower:
            return h.get("value")
    return None


def fetch_recent_messages(max_results: int = 5, query: str | None = None, verbose: bool = False):
    try:
        all_msgs = []
        next_page_token = None

        while len(all_msgs) < max_results:
            remaining = max_results - len(all_msgs)
            fetch_count = min(500, remaining)
            req = service.users().messages().list(
                userId="me",
                maxResults=fetch_count,
                q=query,
                pageToken=next_page_token,
            )
            resp = req.execute()
            msgs = resp.get("messages", [])
            all_msgs.extend(msgs)

            next_page_token = resp.get("nextPageToken")
            if not next_page_token or len(all_msgs) >= max_results:
                break

        if not all_msgs:
            print("No messages found.")
            return
        if verbose:
            for i, m in enumerate(all_msgs[:max_results], start=1):
                msg = service.users().messages().get(
                    userId="me",
                    id=m["id"],
                    format="metadata",
                    metadataHeaders=["From", "To", "Subject", "Date"],
                ).execute()

                payload = msg.get("payload", {})
                headers = payload.get("headers", [])

                frm = _header(headers, "From")
                subj = _header(headers, "Subject")
                date = _header(headers, "Date")
                snippet = (msg.get("snippet") or "").replace("\n", " ")

                print(f"\n{i}. {subj or '(no subject)'}")
                print(f"   From: {frm}")
                print(f"   Date: {date}")
                print(f"   Snippet: {snippet}")

    except HttpError as e:
        raise RuntimeError(f"Gmail API error: {e}") from e
    return all_msgs

_ = fetch_recent_messages(max_results=9, query="in:inbox (category:primary OR category:updates)",verbose=True)



1. Welcome to your Google Cloud Free Trial
   From: Google Cloud <googlecloud@google.com>
   Date: Mon, 25 May 2026 10:48:57 -0700
   Snippet: Get started fast – and see what&#39;s available on Google Cloud. Google Cloud Go to my console Welcome to Google Cloud Learn the fundamentals with this tutorial - and see what else you can do on Google

2. Thanks for applying to ClickUp! 🦄
   From: ClickUp No Reply <noreply@clickup.com>
   Date: Mon, 25 May 2026 17:41:04 +0000
   Snippet: Hi Alireza, Thank you for applying to ClickUp! We&#39;re excited to have received your application for the Growth Machine Learning Engineer role. Your candidacy is currently under review! If your

3. Thank you for applying to Faire!
   From: no-reply@us.greenhouse-mail.io
   Date: Mon, 25 May 2026 17:27:25 +0000
   Snippet: Hi Alireza, Thanks for applying for our Senior Applied AI/ML Scientist - Listing Quality role! We care about creating a great candidate experience. We&#39;ve all been on the other side of t

In [24]:
import re
import base64
import sqlite3
from datetime import datetime
from typing import Any, Optional

from bs4 import BeautifulSoup
from dateutil import parser as date_parser

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or os.getenv("openai_api_key")
if not OPENAI_API_KEY:
    raise ValueError(
        "Missing OpenAI API key. Add OPENAI_API_KEY=... to your .env (or openai_api_key=...)."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

DB_PATH = os.getenv("JOBTRACKER_DB", "jobtracker.sqlite3")
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
CLASSIFY_BATCH_SIZE = int(os.getenv("OPENAI_CLASSIFY_BATCH_SIZE", "12"))
GMAIL_FETCH_BATCH_SIZE = int(os.getenv("GMAIL_FETCH_BATCH_SIZE", "50"))

print("DB_PATH:", DB_PATH)
print("MODEL:", MODEL)
print("CLASSIFY_BATCH_SIZE:", CLASSIFY_BATCH_SIZE)
print("GMAIL_FETCH_BATCH_SIZE:", GMAIL_FETCH_BATCH_SIZE)


DB_PATH: jobtracker.sqlite3
MODEL: gpt-5-nano
CLASSIFY_BATCH_SIZE: 12
GMAIL_FETCH_BATCH_SIZE: 50


In [25]:
def init_db(db_path: str = DB_PATH) -> None:
    conn = sqlite3.connect(db_path)
    try:
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA foreign_keys=ON;")

        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS emails (
              gmail_message_id TEXT PRIMARY KEY,
              gmail_thread_id TEXT,
              internal_date_ms INTEGER,
              from_addr TEXT,
              to_addr TEXT,
              subject TEXT,
              date_raw TEXT,
              snippet TEXT,
              body_text TEXT,
              created_at TEXT NOT NULL
            );
            """
        )

        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS processed_messages (
              gmail_message_id TEXT PRIMARY KEY,
              category TEXT,
              confidence REAL,
              model TEXT,
              raw_json TEXT,
              processed_at TEXT NOT NULL
            );
            """
        )

        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS applications (
              id INTEGER PRIMARY KEY AUTOINCREMENT,
              app_key TEXT NOT NULL UNIQUE,
              company TEXT,
              job_title TEXT,
              job_id TEXT,
              source TEXT,
              applied_date TEXT,
              status TEXT NOT NULL,
              last_update_date TEXT,
              last_email_message_id TEXT,
              last_email_date_raw TEXT,
              confidence REAL,
              notes TEXT,
              created_at TEXT NOT NULL,
              updated_at TEXT NOT NULL
            );
            """
        )

        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS app_events (
              id INTEGER PRIMARY KEY AUTOINCREMENT,
              app_key TEXT NOT NULL,
              event_type TEXT NOT NULL,
              event_date TEXT,
              gmail_message_id TEXT,
              raw_json TEXT,
              created_at TEXT NOT NULL,
              FOREIGN KEY(app_key) REFERENCES applications(app_key)
            );
            """
        )

        conn.commit()
    finally:
        conn.close()


init_db()
print("DB initialized.")


DB initialized.


In [ ]:
def _b64url_decode(data: str) -> bytes:
    return base64.urlsafe_b64decode(data.encode("utf-8"))


def _extract_bodies(payload: dict) -> dict[str, str]:
    out: dict[str, str] = {}

    def walk(part: dict):
        mime = part.get("mimeType")
        body = part.get("body") or {}
        data = body.get("data")
        if data and mime in ("text/plain", "text/html"):
            try:
                out[mime] = _b64url_decode(data).decode("utf-8", errors="replace")
            except Exception:
                out[mime] = ""

        for p in part.get("parts") or []:
            walk(p)

    walk(payload or {})
    return out


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text("\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_message_full(message_id: str) -> dict:
    return service.users().messages().get(userId="me", id=message_id, format="full").execute()


def get_messages_full_batch(message_ids: list[str]) -> dict[str, dict]:
    """Fetch full message payloads using Gmail batch HTTP (far fewer TCP round trips than sequential get)."""
    # message_ids is a list of strings, where each string is a Gmail message ID to fetch.
    if not message_ids:
        return {}
    out: dict[str, dict] = {}
    chunk_sz = max(1, min(GMAIL_FETCH_BATCH_SIZE, 100))

    for i in range(0, len(message_ids), chunk_sz):
        chunk = message_ids[i : i + chunk_sz]
        batch = service.new_batch_http_request()

        for mid in chunk:

            def _cb(rid, resp, exc, mid=mid):
                if exc is not None:
                    out[mid] = get_message_full(mid)
                else:
                    out[mid] = resp

            req = service.users().messages().get(userId="me", id=mid, format="full")
            batch.add(req, callback=_cb, request_id=mid)

        batch.execute()

    return out


def normalize_date(date_raw: Optional[str]) -> Optional[str]:
    if not date_raw:
        return None
    try:
        dt = date_parser.parse(date_raw)
        return dt.isoformat()
    except Exception:
        return None


In [27]:
JOB_EMAIL_SYSTEM_PROMPT = """You are a precise information extraction system.

You will be given a single email about jobs/careers (or not related).
Your task: decide whether it relates to a job application the user made, and if so, classify the event and extract key fields.

Return ONLY valid JSON matching this schema:
{
  "is_job_related": boolean,
  "category": "application_confirmation" | "rejection" | "follow_up" | "interview" | "offer" | "job_alert" | "newsletter" | "other",
  "company": string | null,
  "job_title": string | null,
  "job_id": string | null,
  "applied_date": string | null,
  "event_date": string | null,
  "confidence": number,
  "reason": string,
  "evidence": {
    "company": string | null,
    "job_title": string | null,
    "job_id": string | null
  },
  "notes": string | null
}

Rules:
- If it is not about a job application process (e.g. grocery promos, receipts), set is_job_related=false.
- If it's about a job posting alert or LinkedIn “add connection” etc, keep is_job_related=true but category="job_alert" or "other".
- Prefer company/job_title/job_id only when clearly supported; otherwise null.
"""


JOB_EMAIL_BATCH_SYSTEM_PROMPT = """You are a precise information extraction system.

The user message is JSON: {"emails": [ ... ]}. Each element has:
- gmail_message_id: string (opaque; copy exactly into your output for that email)
- from, subject, date, snippet, body: same meaning as in the single-email task (body may be truncated)

For EVERY input email, produce one object in "results" with this shape:
{
  "gmail_message_id": string,
  "is_job_related": boolean,
  "category": "application_confirmation" | "rejection" | "follow_up" | "interview" | "offer" | "job_alert" | "newsletter" | "other",
  "company": string | null,
  "job_title": string | null,
  "job_id": string | null,
  "applied_date": string | null,
  "event_date": string | null,
  "confidence": number,
  "reason": string,
  "evidence": {"company": string | null, "job_title": string | null, "job_id": string | null},
  "notes": string | null
}

Return ONLY valid JSON: {"results": [ ... ]}.
Rules:
- results MUST have the SAME LENGTH as input emails, and MUST be in the SAME ORDER.
- gmail_message_id in each result MUST equal the corresponding input gmail_message_id.
- Apply the same classification rules as for single emails (non-job mail, job alerts, etc.).
"""


def classify_email_with_openai(*, subject: str | None, from_addr: str | None, date_raw: str | None, snippet: str | None, body_text: str | None) -> dict[str, Any]:
    user_content = {
        "from": from_addr,
        "subject": subject,
        "date": date_raw,
        "snippet": snippet,
        "body": (body_text or "")[:1500],
    }

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JOB_EMAIL_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_content, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    content = resp.choices[0].message.content or "{}"
    return json.loads(content)


def classify_emails_batch_openai(batch: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    """One API call for len(batch) messages. Returns gmail_message_id -> extraction dict (no id key)."""
    if not batch:
        return {}

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JOB_EMAIL_BATCH_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps({"emails": batch}, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    content = json.loads(resp.choices[0].message.content or "{}")
    results = content.get("results")
    if not isinstance(results, list):
        return {}

    out: dict[str, dict[str, Any]] = {}
    for item in results:
        if not isinstance(item, dict):
            continue
        mid = item.get("gmail_message_id")
        if not mid:
            continue
        extracted = {k: v for k, v in item.items() if k != "gmail_message_id"}
        out[str(mid)] = extracted

    return out


In [14]:
ALLOWED_APP_CATEGORIES = {"application_confirmation", "interview", "rejection"}


def make_app_key(company: Optional[str], job_title: Optional[str], job_id: Optional[str]) -> str:
    c = (company or "").strip().lower()
    t = (job_title or "").strip().lower()
    j = (job_id or "").strip().lower()
    if j:
        return f"job_id:{j}"
    if c or t:
        return f"company_title:{c}|{t}".strip("|")
    return "unknown"


def fetch_processed_message_ids(db_path: str, message_ids: list[str]) -> set[str]:
    """Return which of message_ids are already in processed_messages (batched IN queries)."""
    if not message_ids:
        return set()
    lim = 900  # stay under SQLite max host parameters
    found: set[str] = set()
    conn = sqlite3.connect(db_path)
    try:
        for off in range(0, len(message_ids), lim):
            chunk = message_ids[off : off + lim]
            ph = ",".join("?" * len(chunk))
            rows = conn.execute(
                f"SELECT gmail_message_id FROM processed_messages WHERE gmail_message_id IN ({ph})",
                chunk,
            ).fetchall()
            found.update(r[0] for r in rows)
    finally:
        conn.close()
    return found


def is_message_processed(db_path: str, gmail_message_id: str) -> bool:
    conn = sqlite3.connect(db_path)
    try:
        row = conn.execute(
            "SELECT 1 FROM processed_messages WHERE gmail_message_id = ? LIMIT 1;",
            (gmail_message_id,),
        ).fetchone()
        return row is not None
    finally:
        conn.close()


def mark_message_processed(db_path: str, gmail_message_id: str, extracted: dict[str, Any]) -> None:
    now = datetime.utcnow().isoformat()
    conn = sqlite3.connect(db_path)
    try:
        conn.execute(
            """
            INSERT INTO processed_messages (gmail_message_id, category, confidence, model, raw_json, processed_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(gmail_message_id) DO UPDATE SET
              category=excluded.category,
              confidence=excluded.confidence,
              model=excluded.model,
              raw_json=excluded.raw_json,
              processed_at=excluded.processed_at;
            """,
            (
                gmail_message_id,
                extracted.get("category"),
                extracted.get("confidence"),
                MODEL,
                json.dumps(extracted, ensure_ascii=False),
                now,
            ),
        )
        conn.commit()
    finally:
        conn.close()


def upsert_email_and_event(
    *,
    db_path: str,
    gmail_message_id: str,
    gmail_thread_id: Optional[str],
    internal_date_ms: Optional[int],
    from_addr: Optional[str],
    to_addr: Optional[str],
    subject: Optional[str],
    date_raw: Optional[str],
    snippet: Optional[str],
    body_text: Optional[str],
    extracted: dict[str, Any],
) -> None:
    now = datetime.utcnow().isoformat()

    status = extracted.get("category") or "other"
    if status not in ALLOWED_APP_CATEGORIES:
        return

    app_key = make_app_key(extracted.get("company"), extracted.get("job_title"), extracted.get("job_id"))
    conf = extracted.get("confidence")

    conn = sqlite3.connect(db_path)
    try:
        conn.execute(
            """
            INSERT INTO emails (
              gmail_message_id, gmail_thread_id, internal_date_ms,
              from_addr, to_addr, subject, date_raw, snippet, body_text, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(gmail_message_id) DO UPDATE SET
              gmail_thread_id=excluded.gmail_thread_id,
              internal_date_ms=excluded.internal_date_ms,
              from_addr=excluded.from_addr,
              to_addr=excluded.to_addr,
              subject=excluded.subject,
              date_raw=excluded.date_raw,
              snippet=excluded.snippet,
              body_text=excluded.body_text;
            """,
            (
                gmail_message_id,
                gmail_thread_id,
                internal_date_ms,
                from_addr,
                to_addr,
                subject,
                date_raw,
                snippet,
                body_text,
                now,
            ),
        )

        applied_date = extracted.get("applied_date")
        event_date = extracted.get("event_date") or normalize_date(date_raw)

        conn.execute(
            """
            INSERT INTO applications (
              app_key, company, job_title, job_id, source, applied_date,
              status, last_update_date, last_email_message_id, last_email_date_raw,
              confidence, notes, created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_key) DO UPDATE SET
              company=COALESCE(excluded.company, applications.company),
              job_title=COALESCE(excluded.job_title, applications.job_title),
              job_id=COALESCE(excluded.job_id, applications.job_id),
              status=excluded.status,
              last_update_date=excluded.last_update_date,
              last_email_message_id=excluded.last_email_message_id,
              last_email_date_raw=excluded.last_email_date_raw,
              confidence=excluded.confidence,
              notes=excluded.notes,
              updated_at=excluded.updated_at;
            """,
            (
                app_key,
                extracted.get("company"),
                extracted.get("job_title"),
                extracted.get("job_id"),
                from_addr,
                applied_date,
                status,
                event_date,
                gmail_message_id,
                date_raw,
                conf,
                extracted.get("notes") or extracted.get("reason"),
                now,
                now,
            ),
        )

        conn.execute(
            """
            INSERT INTO app_events (app_key, event_type, event_date, gmail_message_id, raw_json, created_at)
            VALUES (?, ?, ?, ?, ?, ?);
            """,
            (
                app_key,
                status,
                event_date,
                gmail_message_id,
                json.dumps(extracted, ensure_ascii=False),
                now,
            ),
        )

        conn.commit()
    finally:
        conn.close()


# Core (batch OpenAI)


In [69]:
def classify_emails_batch_openai(batch: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    """One API call for len(batch) messages. Returns gmail_message_id -> extraction dict (no id key)."""
    if not batch:
        return {}

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JOB_EMAIL_BATCH_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps({"emails": batch}, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    content = json.loads(resp.choices[0].message.content or "{}")
    results = content.get("results")
    if not isinstance(results, list):
        return {}

    out: dict[str, dict[str, Any]] = {}
    for item in results:
        if not isinstance(item, dict):
            continue
        mid = item.get("gmail_message_id")
        if not mid:
            continue
        extracted = {k: v for k, v in item.items() if k != "gmail_message_id"}
        out[str(mid)] = extracted

    return out


def get_messages_full_batch(message_ids: list[str]) -> dict[str, dict]:
    """Fetch full message payloads using Gmail batch HTTP (far fewer TCP round trips than sequential get)."""
    if not message_ids:
        return {}
    out: dict[str, dict] = {}
    chunk_sz = max(1, min(GMAIL_FETCH_BATCH_SIZE, 100))

    for i in range(0, len(message_ids), chunk_sz):
        chunk = message_ids[i : i + chunk_sz]
        batch = service.new_batch_http_request()

        for mid in chunk:

            def _cb(rid, resp, exc, mid=mid):
                if exc is not None:
                    out[mid] = get_message_full(mid)
                else:
                    out[mid] = resp

            req = service.users().messages().get(userId="me", id=mid, format="full")
            batch.add(req, callback=_cb, request_id=mid)

        batch.execute()

    return out

class InboxBatchProcessor:
    def __init__(self, 
                 max_results: int = 50, 
                 query: str = "in:inbox", 
                 force: bool = False,
                 batch_size: int | None = None):

        self.max_results = max_results
        self.query = query
        self.force = force
        self.bs = batch_size if batch_size is not None else CLASSIFY_BATCH_SIZE
        if self.bs < 1:
            raise ValueError("batch_size must be >= 1")

        self.msgs = []
        self.id_to_index = {}
        self.pending_ids = []
        self.skipped = 0
        self.full_by_id = {}
        self.pending = []
        self.processed = 0
        self.stored = 0
        self.total = 0

    def fetch_messages(self):
        self.msgs = fetch_recent_messages(max_results=self.max_results, query=self.query)
        if not self.msgs:
            print("No messages to process.")
            self.total = 0
            return self
        print(f"Found {len(self.msgs)} messages.")
        self.id_to_index = {m["id"]: i for i, m in enumerate(self.msgs, start=1)}
        self.total = len(self.msgs)
        all_ids = [m["id"] for m in self.msgs]
        if self.force:
            already: set[str] = set()
            self.skipped = 0
        else:
            already = fetch_processed_message_ids(DB_PATH, all_ids)
            self.skipped = len(already)
        self.pending_ids = [mid for mid in all_ids if self.force or mid not in already]
        return self

    def fetch_full_messages(self):
        print(f"Gmail: fetching {len(self.pending_ids)} full messages (batch up to {GMAIL_FETCH_BATCH_SIZE} per HTTP request)…")
        self.full_by_id = get_messages_full_batch(self.pending_ids)
        return self

    def prepare_pending(self):
        self.pending.clear()
        for mid in self.pending_ids:
            full = self.full_by_id[mid]
            payload = full.get("payload") or {}
            headers = payload.get("headers") or []
            from_addr = _header(headers, "From")
            to_addr = _header(headers, "To")
            subject = _header(headers, "Subject")
            date_raw = _header(headers, "Date")
            snippet = (full.get("snippet") or "").replace("\n", " ")
            bodies = _extract_bodies(payload)
            body_text = bodies.get("text/plain")
            if not body_text and bodies.get("text/html"):
                body_text = html_to_text(bodies["text/html"])
            self.pending.append(
                {
                    "message_id": mid,
                    "full": full,
                    "from_addr": from_addr,
                    "to_addr": to_addr,
                    "subject": subject,
                    "date_raw": date_raw,
                    "snippet": snippet,
                    "body_text": body_text,
                }
            )
        print(f"To classify now: {len(self.pending)} (skipped already-processed: {self.skipped})")
        return self

    def classify_and_store(self):
        for start in range(0, len(self.pending), self.bs):
            # self.pending is a list of emails that still need to be processed (not yet classified or stored).
            # chunk is a slice of pending emails to process in this batch.
            chunk = self.pending[start: start + self.bs]
            api_batch = [
                {
                    "gmail_message_id": row["message_id"],
                    "from": row["from_addr"],
                    "subject": row["subject"],
                    "date": row["date_raw"],
                    "snippet": row["snippet"],
                    "body": (row["body_text"] or "")[:1500],
                }
                for row in chunk
            ]
            by_id = classify_emails_batch_openai(api_batch)
   
   

            for row in chunk:
                mid = row["message_id"]
                extracted = by_id.get(mid)
                if extracted is None:
                    extracted = classify_email_with_openai(
                        subject=row["subject"],
                        from_addr=row["from_addr"],
                        date_raw=row["date_raw"],
                        snippet=row["snippet"],
                        body_text=row["body_text"],
                    )
                mark_message_processed(DB_PATH, mid, extracted)
                self.processed += 1

                full = row["full"]
                upsert_email_and_event(
                    db_path=DB_PATH,
                    gmail_message_id=mid,
                    gmail_thread_id=full.get("threadId"),
                    internal_date_ms=int(full["internalDate"]) if full.get("internalDate") else None,
                    from_addr=row["from_addr"],
                    to_addr=row["to_addr"],
                    subject=row["subject"],
                    date_raw=row["date_raw"],
                    snippet=row["snippet"],
                    body_text=row["body_text"],
                    extracted=extracted,
                )
                if extracted.get("category") in ALLOWED_APP_CATEGORIES:
                    self.stored += 1

                idx = self.id_to_index.get(mid, self.processed)
                cat = extracted.get("category")
                comp = extracted.get("company")
                title = extracted.get("job_title")
                print(f"[{idx}/{self.total}] {cat} | {comp} | {title} | subj={row['subject']!r}")
        return self

    def run(self):
        if not self.fetch_messages().msgs:
            return self
        return self.fetch_full_messages().prepare_pending().classify_and_store().summarize()

    def summarize(self):
        print(f"Done. skipped={self.skipped} processed_now={self.processed} stored_app_records={self.stored}")
        return self


# Example usage
ali = InboxBatchProcessor(
    max_results=15,
    query="in:inbox (category:primary OR category:updates)",
    force=True,
)

ali.fetch_messages().fetch_full_messages().prepare_pending()#.prepare_pending()



Found 15 messages.
Gmail: fetching 15 full messages (batch up to 50 per HTTP request)…
To classify now: 15 (skipped already-processed: 0)


In [70]:

api_batch = [
                {
                    "gmail_message_id": row["message_id"],
                    "from": row["from_addr"],
                    "subject": row["subject"],
                    "date": row["date_raw"],
                    "snippet": row["snippet"],
                    "body": (row["body_text"] or "")[:600],
                }
                for row in ali.pending
            ]
result = classify_emails_batch_openai(api_batch)


In [71]:
result

{'19e60411133d4a83': {'is_job_related': False,
  'category': 'newsletter',
  'company': 'Google Cloud',
  'job_title': None,
  'job_id': None,
  'applied_date': None,
  'event_date': None,
  'confidence': 0.6,
  'reason': 'General welcome/marketing email; not related to job applications.',
  'evidence': {'company': 'Google Cloud', 'job_title': None, 'job_id': None},
  'notes': None},
 '19e6039dbcf948b1': {'is_job_related': True,
  'category': 'application_confirmation',
  'company': 'ClickUp',
  'job_title': 'Growth Machine Learning Engineer',
  'job_id': None,
  'applied_date': '2026-05-25',
  'event_date': None,
  'confidence': 0.85,
  'reason': 'Application received; candidate is under review.',
  'evidence': {'company': 'ClickUp',
   'job_title': 'Growth Machine Learning Engineer',
   'job_id': None},
  'notes': None},
 '19e602d5eae9b2ee': {'is_job_related': True,
  'category': 'application_confirmation',
  'company': 'Faire',
  'job_title': 'Senior Applied AI/ML Scientist - Listin

In [45]:
def fetch_applications(status: str | None = None, company_like: str | None = None):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        sql = "SELECT * FROM applications WHERE 1=1"
        params: list[Any] = []
        if status:
            sql += " AND status = ?"
            params.append(status)
        if company_like:
            sql += " AND company LIKE ?"
            params.append(f"%{company_like}%")
        sql += " ORDER BY updated_at DESC LIMIT 100"

        rows = conn.execute(sql, params).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


apps = fetch_applications()
apps[:20]


[{'id': 358,
  'app_key': 'company_title:opendoor labs inc.|applied scientist',
  'company': 'Opendoor Labs Inc.',
  'job_title': 'Applied Scientist',
  'job_id': None,
  'source': 'Opendoor <no-reply@ats.rippling.com>',
  'applied_date': None,
  'status': 'application_confirmation',
  'last_update_date': '2026-05-25T15:12:08Z',
  'last_email_message_id': '19e5fb187609f489',
  'last_email_date_raw': 'Mon, 25 May 2026 15:12:08 +0000 (UTC)',
  'confidence': 0.8,
  'notes': 'Email confirms receipt of job application for Applied Scientist at Opendoor Labs Inc.',
  'created_at': '2026-05-25T18:08:52.195703',
  'updated_at': '2026-05-25T18:08:52.195703'},
 {'id': 357,
  'app_key': 'job_id:54477',
  'company': 'TELUS',
  'job_title': 'Data Scientist - Analytics & AI (TELUS Health)',
  'job_id': '54477',
  'source': '"People & Culture at TELUS" <noreply@telus.com>',
  'applied_date': None,
  'status': 'application_confirmation',
  'last_update_date': '2026-05-25T15:32:09Z',
  'last_email_messa

In [32]:

def update_application_status(app_key: str, new_status: str, note: str | None = None):
    now = datetime.utcnow().isoformat()
    conn = sqlite3.connect(DB_PATH)
    try:
        conn.execute(
            """
            UPDATE applications
            SET status = ?, notes = COALESCE(?, notes), updated_at = ?
            WHERE app_key = ?;
            """,
            (new_status, note, now, app_key),
        )

        conn.execute(
            """
            INSERT INTO app_events (app_key, event_type, event_date, gmail_message_id, raw_json, created_at)
            VALUES (?, ?, ?, NULL, ?, ?);
            """,
            (
                app_key,
                f"manual:{new_status}",
                now,
                json.dumps({"note": note, "status": new_status}, ensure_ascii=False),
                now,
            ),
        )

        conn.commit()
    finally:
        conn.close()


In [33]:
def get_application_timeline(app_key: str):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        rows = conn.execute(
            """
            SELECT event_type, event_date, gmail_message_id, created_at
            FROM app_events
            WHERE app_key = ?
            ORDER BY created_at ASC;
            """,
            (app_key,),
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


In [36]:
# Pandas + FAISS view (same as gmail.ipynb). Run `process_inbox_to_db_batch` first if the table is empty.

# %pip -q install faiss-cpu pandas scikit-learn

import pandas as pd
import numpy as np
import sqlite3

import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
try:
    apps = conn.execute(
        """
        SELECT app_key, company, job_title, job_id, applied_date, status, updated_at
        FROM applications
        ORDER BY updated_at DESC;
        """
    ).fetchall()

    if not apps:
        raise RuntimeError("No rows in applications table yet. Run process_inbox_to_db_batch() first.")

    apps_df = pd.DataFrame([dict(r) for r in apps])

    events = conn.execute(
        """
        SELECT app_key, event_type, event_date
        FROM app_events
        WHERE event_type IN ('application_confirmation', 'rejection', 'interview')
        ;
        """
    ).fetchall()

finally:
    conn.close()

events_df = pd.DataFrame([dict(r) for r in events])

apply_dates = (
    events_df[events_df["event_type"] == "application_confirmation"]
    .groupby("app_key")["event_date"]
    .min()
    .rename("date_of_applied")
    .reset_index()
)

rejection_dates = (
    events_df[events_df["event_type"] == "rejection"]
    .groupby("app_key")["event_date"]
    .min()
    .rename("date_of_rejection")
    .reset_index()
)

out = apps_df.merge(apply_dates, on="app_key", how="left").merge(rejection_dates, on="app_key", how="left")

out["status"] = np.where(out["date_of_rejection"].notna(), "rejection", out["status"])

texts = (out["company"].fillna("") + " " + out["job_title"].fillna("")).tolist()
vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
X = vectorizer.fit_transform(texts).astype(np.float32)
X_dense = X.toarray().astype(np.float32)

index = faiss.IndexFlatL2(X_dense.shape[1])
index.add(X_dense)

D, I = index.search(X_dense, 1)
out["faiss_self_match_idx"] = I[:, 0]
out["faiss_self_distance"] = D[:, 0]

final_cols = [
    "company",
    "job_title",
    "date_of_applied",
    "date_of_rejection",
    "status",
]

out["date_of_applied"] = out["date_of_applied"].fillna(out["applied_date"])

df = out[final_cols].sort_values(by="date_of_applied", ascending=False, na_position="last")

df.sort_values(by="date_of_applied", ascending=False, na_position="last")


,company,job_title,date_of_applied,date_of_rejection,status
13,Scotiabank,Senior Data Scientist,2026-05-15T15:30:13+00:00,NaN,application_confirmation
6,TD,Data Scientist III,2026-05-15,NaN,application_confirmation
8,Blanc Labs,Senior Data Scientist – AI & Agentic,2026-05-15,NaN,application_confirmation
9,PolyAI,Forward Deployed AI Engineer,2026-05-15,NaN,application_confirmation
12,Robots & Pencils,Senior AI Engineer,2026-05-15,NaN,application_confirmation
...,...,...,...,...,...
239,BDO Canada LLP,AI Engineer,NaN,2026-05-03T07:54:20Z,rejection
244,Cantire,Senior Data Scientist,NaN,NaN,interview
245,"Intuition Machines, Inc.",Senior/Lead ML Applied Scientist,NaN,2026-05-06T09:06:10Z,rejection
246,Copoly.ai,Machine Learning Bioinformatics Engineer,NaN,2026-05-07T15:40:05Z,rejection


In [20]:
df_out = df.copy()

def standardize_date_column(series):
    result = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    needs_fallback = result.isna() & series.notna()
    if needs_fallback.any():
        fallback = pd.to_datetime(series[needs_fallback], errors="coerce", format="%Y-%m-%d", utc=True)
        result.loc[needs_fallback] = fallback
    return result.dt.strftime("%Y-%m-%d")

df_out["date_of_applied"] = standardize_date_column(df_out["date_of_applied"])
df_out["date_of_rejection"] = standardize_date_column(df_out["date_of_rejection"])
df_out = df_out.sort_values(by="date_of_applied", ascending=True, na_position="last").reset_index(drop=True)
df_out.to_csv("applications_with_dates.csv", index=False)
